In [ ]:
"""
FDA NDC Directory Loader (Product + Package)
--------------------------------------------
Downloads the daily NDC text files from FDA,
builds standardized labeler_id and 11-digit NDC,
and writes both parquet and Excel outputs.
Excel NDC columns are forced to text to preserve leading zeros.
"""

import pandas as pd
import requests
import zipfile
import io
from pathlib import Path
from datetime import datetime
from config import out_folder

out_path = Path(out_folder)
out_path.mkdir(parents=True, exist_ok=True)

# Download text zip
url = "https://www.accessdata.fda.gov/cder/ndctext.zip"
print("Downloading NDC text zip...")
response = requests.get(url)
response.raise_for_status()

with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    print("Files in zip:")
    for name in z.namelist():
        print(" ", name)

    # --- PACKAGE ---
    package_file = [n for n in z.namelist() if "package" in n.lower() and n.endswith(".txt")][0]
    with z.open(package_file) as f:
        df_package = pd.read_csv(f, sep="\t", dtype=str, low_memory=False, encoding="latin-1")

    # --- PRODUCT ---
    product_file = [n for n in z.namelist() if "product" in n.lower() and n.endswith(".txt")][0]
    with z.open(product_file) as f:
        df_product = pd.read_csv(f, sep="\t", dtype=str, low_memory=False, encoding="latin-1")

# ---------- Helpers ----------
def build_labeler(product_ndc):
    return product_ndc.astype(str).str.split("-").str[0].str.zfill(5)

def build_ndc11(ndc):
    parts = ndc.astype(str).str.split("-", expand=True)
    if parts.shape[1] == 3:
        labeler  = parts[0].str.zfill(5)
        product  = parts[1].str.zfill(4)
        package  = parts[2].str.zfill(2)
        return labeler + product + package
    return pd.Series([None] * len(ndc))

# ---------- Add columns ----------
load_ts = datetime.now()

df_package["labeler_id"] = build_labeler(df_package["PRODUCTNDC"])
df_package["ndc11"]      = build_ndc11(df_package["NDCPACKAGECODE"])
df_package["load_date"]  = load_ts

df_product["labeler_id"] = build_labeler(df_product["PRODUCTNDC"])
df_product["load_date"]  = load_ts

# ---------- Save ----------

# Comment Out xlsx files only use for viewability
df_package.to_excel(out_path / "ndc_package.xlsx", index=False)
df_package.to_parquet(out_path / "ndc_package.parquet", index=False)

# Comment Out xlsx files only use for viewability
df_product.to_excel(out_path / "ndc_product.xlsx", index=False)
df_product.to_parquet(out_path / "ndc_product.parquet", index=False)

print(f"Package: {df_package.shape[0]:,} rows → ndc_package.xlsx / .parquet")
print(f"Product: {df_product.shape[0]:,} rows → ndc_product.xlsx / .parquet")
print("Done.")

In [ ]:
"""
FDA Compounded Drugs NDC Directory Loader
-----------------------------------------
Downloads the daily compounders NDC file from FDA,
builds standardized labeler_id and 11-digit NDC,
and writes both parquet and Excel outputs.
Excel NDC columns are forced to text to preserve leading zeros.
"""

from pathlib import Path
from datetime import datetime
import zipfile
import requests
import pandas as pd
import openpyxl

from config import out_folder

out_path = Path(out_folder)
out_path.mkdir(parents=True, exist_ok=True)

# ---------- Download ----------
url = "https://www.accessdata.fda.gov/cder/compounders_ndc_directory.zip"
zip_path = out_path / "compounders_ndc_directory.zip"

print("Downloading compound NDC file...")
resp = requests.get(url, timeout=120)
resp.raise_for_status()
zip_path.write_bytes(resp.content)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extract("compounders_ndc_directory.txt", out_path)

txt_path = out_path / "compounders_ndc_directory.txt"

# ---------- Load ----------
df = pd.read_csv(txt_path, sep="\t", dtype=str, low_memory=False)
print(f"Loaded {len(df):,} rows")

# ---------- Helpers ----------
def build_labeler(product_ndc):
    return product_ndc.astype(str).str.split("-").str[0].str.zfill(5)

def build_ndc11(ndc_package_code):
    parts = ndc_package_code.astype(str).str.split("-")
    labeler = parts.str[0].str.zfill(5)
    product = parts.str[1].str.zfill(4)
    package = parts.str[2].str.zfill(2)
    return labeler + product + package

df["labeler_id"] = build_labeler(df["PRODUCTNDC"])
df["ndc11"] = build_ndc11(df["NDCPACKAGECODE"])
df["load_date"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# ---------- Save ----------
xlsx_path = out_path / "ndc_compound.xlsx"
parquet_path = out_path / "ndc_compound.parquet"

df.to_excel(xlsx_path, index=False)
df.to_parquet(parquet_path, index=False)

# Force NDC columns to text in Excel
def force_text_columns(filepath, columns):
    wb = openpyxl.load_workbook(filepath)
    ws = wb.active
    headers = [cell.value for cell in ws[1]]
    for col_name in columns:
        if col_name in headers:
            col_idx = headers.index(col_name) + 1
            for cell in ws.iter_cols(min_col=col_idx, max_col=col_idx, min_row=2):
                for c in cell:
                    c.number_format = "@"
                    if c.value is not None:
                        c.value = str(c.value)
    wb.save(filepath)

force_text_columns(xlsx_path, ["PRODUCTNDC", "NDCPACKAGECODE", "labeler_id", "ndc11"])

print(f"Compound: {df.shape[0]:,} rows → ndc_compound.xlsx / .parquet")
print("Done.")

In [ ]:
"""
FDA Unfinished Drugs NDC Directory Loader
-----------------------------------------
Downloads the daily unfinished NDC files from FDA,
builds standardized labeler_id (both files) and 11-digit NDC (package only),
and writes both parquet and Excel outputs.
Excel NDC columns are forced to text to preserve leading zeros.
"""

from pathlib import Path
from datetime import datetime
import zipfile
import requests
import pandas as pd
import openpyxl

from config import out_folder

out_path = Path(out_folder)
out_path.mkdir(parents=True, exist_ok=True)
print(f"Output folder: {out_path.resolve()}")

# ---------- Download ----------
url = "https://www.accessdata.fda.gov/cder/ndc_unfinished.zip"
zip_path = out_path / "ndc_unfinished.zip"

print("Downloading unfinished NDC files...")
resp = requests.get(url, timeout=120)
resp.raise_for_status()
zip_path.write_bytes(resp.content)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extract("unfinished_product.txt", out_path)
    z.extract("unfinished_package.txt", out_path)
    product_modified = datetime(*z.getinfo("unfinished_product.txt").date_time).strftime("%Y-%m-%d %H:%M:%S")
    package_modified = datetime(*z.getinfo("unfinished_package.txt").date_time).strftime("%Y-%m-%d %H:%M:%S")

# ---------- Load (encoding-safe) ----------
def safe_read(path):
    for enc in ("utf-8", "latin-1", "cp1252"):
        try:
            return pd.read_csv(path, sep="\t", dtype=str, low_memory=False, encoding=enc)
        except Exception as e:
            print(f"  {enc} failed: {e}")
    raise RuntimeError(f"Could not read {path}")

print("Loading product...")
df_product = safe_read(out_path / "unfinished_product.txt")
print("Loading package...")
df_package = safe_read(out_path / "unfinished_package.txt")

print(f"Product: {len(df_product):,} rows")
print(f"Package: {len(df_package):,} rows")

# ---------- Helpers ----------
def build_labeler(product_ndc):
    return product_ndc.astype(str).str.split("-").str[0].str.zfill(5)

def build_ndc11(ndc_package_code):
    parts = ndc_package_code.astype(str).str.split("-")
    labeler = parts.str[0].str.zfill(5)
    product = parts.str[1].str.zfill(4)
    package = parts.str[2].str.zfill(2)
    return labeler + product + package

# ---------- Enrich ----------
df_product["labeler_id"] = build_labeler(df_product["PRODUCTNDC"])
df_product["load_date"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

df_package["labeler_id"] = build_labeler(df_package["PRODUCTNDC"])
df_package["ndc11"] = build_ndc11(df_package["NDCPACKAGECODE"])
df_package["load_date"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# ---------- Save ----------
product_xlsx = out_path / "ndc_unfinished_product.xlsx"
package_xlsx = out_path / "ndc_unfinished_package.xlsx"

df_product.to_excel(product_xlsx, index=False)
df_product.to_parquet(out_path / "ndc_unfinished_product.parquet", index=False)

df_package.to_excel(package_xlsx, index=False)
df_package.to_parquet(out_path / "ndc_unfinished_package.parquet", index=False)

# Force NDC columns to text in Excel
def force_text_columns(filepath, columns):
    wb = openpyxl.load_workbook(filepath)
    ws = wb.active
    headers = [cell.value for cell in ws[1]]
    for col_name in columns:
        if col_name in headers:
            col_idx = headers.index(col_name) + 1
            for cell in ws.iter_cols(min_col=col_idx, max_col=col_idx, min_row=2):
                for c in cell:
                    c.number_format = "@"
                    if c.value is not None:
                        c.value = str(c.value)
    wb.save(filepath)

force_text_columns(product_xlsx, ["PRODUCTNDC", "labeler_id"])
force_text_columns(package_xlsx, ["PRODUCTNDC", "NDCPACKAGECODE", "labeler_id", "ndc11"])

print(f"\nFiles written to: {out_path.resolve()}")
print(f"  {product_xlsx.name}")
print(f"  ndc_unfinished_product.parquet")
print(f"  {package_xlsx.name}")
print(f"  ndc_unfinished_package.parquet")
print("Done.")

In [ ]:
"""
FDA Excluded Drugs NDC Directory Loader
---------------------------------------
Downloads the daily excluded NDC files from FDA,
builds standardized labeler_id (both files) and 11-digit NDC (package only),
and writes both parquet and Excel outputs.
Excel NDC columns are forced to text to preserve leading zeros.
"""

from pathlib import Path
from datetime import datetime
import zipfile
import requests
import pandas as pd
import openpyxl

from config import out_folder

out_path = Path(out_folder)
out_path.mkdir(parents=True, exist_ok=True)
print(f"Output folder: {out_path.resolve()}")

# ---------- Download ----------
url = "https://www.accessdata.fda.gov/cder/ndc_excluded.zip"
zip_path = out_path / "ndc_excluded.zip"

print("Downloading excluded NDC files...")
resp = requests.get(url, timeout=180)
resp.raise_for_status()
zip_path.write_bytes(resp.content)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extract("Products_excluded.txt", out_path)
    z.extract("Packages_excluded.txt", out_path)
    product_modified = datetime(*z.getinfo("Products_excluded.txt").date_time).strftime("%Y-%m-%d %H:%M:%S")
    package_modified = datetime(*z.getinfo("Packages_excluded.txt").date_time).strftime("%Y-%m-%d %H:%M:%S")

# ---------- Load (encoding-safe) ----------
def safe_read(path):
    for enc in ("utf-8", "latin-1", "cp1252"):
        try:
            return pd.read_csv(path, sep="\t", dtype=str, low_memory=False, encoding=enc)
        except Exception as e:
            print(f"  {enc} failed: {e}")
    raise RuntimeError(f"Could not read {path}")

print("Loading product...")
df_product = safe_read(out_path / "Products_excluded.txt")
print("Loading package...")
df_package = safe_read(out_path / "Packages_excluded.txt")

print(f"Product: {len(df_product):,} rows")
print(f"Package: {len(df_package):,} rows")

# ---------- Helpers ----------
def build_labeler(product_ndc):
    return product_ndc.astype(str).str.split("-").str[0].str.zfill(5)

def build_ndc11(ndc_package_code):
    parts = ndc_package_code.astype(str).str.split("-")
    labeler = parts.str[0].str.zfill(5)
    product = parts.str[1].str.zfill(4)
    package = parts.str[2].str.zfill(2)
    return labeler + product + package

# ---------- Enrich ----------
df_product["labeler_id"] = build_labeler(df_product["PRODUCTNDC"])
df_product["load_date"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

df_package["labeler_id"] = build_labeler(df_package["PRODUCTNDC"])
df_package["ndc11"] = build_ndc11(df_package["NDCPACKAGECODE"])
df_package["load_date"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# ---------- Save ----------
product_xlsx = out_path / "ndc_excluded_product.xlsx"
package_xlsx = out_path / "ndc_excluded_package.xlsx"

df_product.to_excel(product_xlsx, index=False)
df_product.to_parquet(out_path / "ndc_excluded_product.parquet", index=False)

df_package.to_excel(package_xlsx, index=False)
df_package.to_parquet(out_path / "ndc_excluded_package.parquet", index=False)

# Force NDC columns to text in Excel
def force_text_columns(filepath, columns):
    wb = openpyxl.load_workbook(filepath)
    ws = wb.active
    headers = [cell.value for cell in ws[1]]
    for col_name in columns:
        if col_name in headers:
            col_idx = headers.index(col_name) + 1
            for cell in ws.iter_cols(min_col=col_idx, max_col=col_idx, min_row=2):
                for c in cell:
                    c.number_format = "@"
                    if c.value is not None:
                        c.value = str(c.value)
    wb.save(filepath)

force_text_columns(product_xlsx, ["PRODUCTNDC", "labeler_id"])
force_text_columns(package_xlsx, ["PRODUCTNDC", "NDCPACKAGECODE", "labeler_id", "ndc11"])

print(f"\nFiles written to: {out_path.resolve()}")
print(f"  {product_xlsx.name}")
print(f"  ndc_excluded_product.parquet")
print(f"  {package_xlsx.name}")
print(f"  ndc_excluded_package.parquet")
print("Done.")

In [ ]:
### Quality Check: Core vs Non-Core Overlaps
"""
- Checks how many PRODUCTNDCs from unfinished, excluded, and compound
  also appear in the core product file
- Optional package-level checks on ndc11 (uncomment if desired)
- Uses parquet files already written to out_folder
"""

import duckdb
from pathlib import Path
from config import out_folder

con = duckdb.connect()
out_path = Path(out_folder)

# Load core tables
con.execute(f"""
CREATE OR REPLACE TABLE product AS 
SELECT * FROM read_parquet('{out_path / "ndc_product.parquet"}');

CREATE OR REPLACE TABLE package AS 
SELECT * FROM read_parquet('{out_path / "ndc_package.parquet"}');
""")

# Load non-core tables (add/remove as needed)
con.execute(f"""
CREATE OR REPLACE TABLE unfinished_product AS 
SELECT * FROM read_parquet('{out_path / "ndc_unfinished_product.parquet"}');

CREATE OR REPLACE TABLE excluded_product AS 
SELECT * FROM read_parquet('{out_path / "ndc_excluded_product.parquet"}');

CREATE OR REPLACE TABLE compound AS 
SELECT * FROM read_parquet('{out_path / "ndc_compound.parquet"}');

CREATE OR REPLACE TABLE unfinished_package AS 
SELECT * FROM read_parquet('{out_path / "ndc_unfinished_package.parquet"}');

CREATE OR REPLACE TABLE excluded_package AS 
SELECT * FROM read_parquet('{out_path / "ndc_excluded_package.parquet"}');
""")

print("Tables loaded.\n")

# --- Product-level overlaps (PRODUCTNDC) ---
print("=== PRODUCT OVERLAPS ===")

unfinished_in_core = con.execute("""
SELECT COUNT(DISTINCT u.PRODUCTNDC) 
FROM unfinished_product u
INNER JOIN product p ON u.PRODUCTNDC = p.PRODUCTNDC
""").fetchone()[0]
print(f"Unfinished products found in core product file: {unfinished_in_core:,}")

excluded_in_core = con.execute("""
SELECT COUNT(DISTINCT e.PRODUCTNDC) 
FROM excluded_product e
INNER JOIN product p ON e.PRODUCTNDC = p.PRODUCTNDC
""").fetchone()[0]
print(f"Excluded products found in core product file: {excluded_in_core:,}")

compound_in_core = con.execute("""
SELECT COUNT(DISTINCT c.PRODUCTNDC) 
FROM compound c
INNER JOIN product p ON c.PRODUCTNDC = p.PRODUCTNDC
""").fetchone()[0]
print(f"Compound products found in core product file: {compound_in_core:,}")

# --- Package-level overlaps (ndc11) ---
print("\n=== PACKAGE OVERLAPS ===")

unfinished_pkg_in_core = con.execute("""
SELECT COUNT(DISTINCT u.ndc11) 
FROM unfinished_package u
INNER JOIN package p ON u.ndc11 = p.ndc11
""").fetchone()[0]
print(f"Unfinished packages found in core package file: {unfinished_pkg_in_core:,}")

excluded_pkg_in_core = con.execute("""
SELECT COUNT(DISTINCT e.ndc11) 
FROM excluded_package e
INNER JOIN package p ON e.ndc11 = p.ndc11
""").fetchone()[0]
print(f"Excluded packages found in core package file: {excluded_pkg_in_core:,}")

print("\nDone.")

In [ ]:
# Load excluded package (adjust filename if needed)
con.execute("DROP TABLE IF EXISTS excluded_package")
con.execute(f"""
CREATE TABLE excluded_package AS 
SELECT * FROM read_parquet('{out_path / "ndc_excluded_package.parquet"}');
""")

query = """
SELECT 
    p.ndc11,
    p.NDCPACKAGECODE,
    COALESCE(p.PACKAGEDESCRIPTION, '')          AS package_description,
    p.STARTMARKETINGDATE                        AS package_start_date,
    COALESCE(p.ENDMARKETINGDATE, '')            AS package_end_date,
    p.labeler_id,
    p.load_date,
    
    pr.PRODUCTNDC,
    COALESCE(pr.PROPRIETARYNAME, '')            AS proprietary_name,
    COALESCE(pr.NONPROPRIETARYNAME, '')         AS nonproprietary_name,
    COALESCE(pr.DOSAGEFORMNAME, '')             AS dosage_form,
    COALESCE(pr.ROUTENAME, '')                  AS route,
    COALESCE(pr.SUBSTANCENAME, '')              AS substance,
    COALESCE(pr.ACTIVE_NUMERATOR_STRENGTH, '')  AS strength,
    COALESCE(pr.ACTIVE_INGRED_UNIT, '')         AS strength_unit,
    COALESCE(pr.PHARM_CLASSES, '')              AS pharm_classes,
    COALESCE(pr.DEASCHEDULE, '')                AS dea_schedule,
    COALESCE(pr.LABELERNAME, '')                AS labeler_name,
    COALESCE(pr.MARKETINGCATEGORYNAME, '')      AS marketing_category,
    pr.STARTMARKETINGDATE                       AS product_start_date,
    COALESCE(pr.ENDMARKETINGDATE, '')           AS product_end_date,


FROM package p
LEFT JOIN product pr 
    ON p.PRODUCTID = pr.PRODUCTID
"""

df = con.execute(query).df()

def widen_display(max_colwidth=80, width=1000):
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', width)
    pd.set_option('display.max_colwidth', max_colwidth)

# Usage
widen_display()
print(df.head(10).to_string())
# print(f"Joined rows: {len(df):,}")
# print(df.head(10))
# print(f"Excluded packages found in core: {df['is_excluded'].sum():,}")
# Export to Excel in the output folder
# output_file = out_path / "ndc_package_product_with_excluded_flag.xlsx"
# df.to_excel(output_file, index=False)

# print(f"Exported to: {output_file}")

In [ ]:
# =============================================================================
# RxNorm NDC Crosswalk
# =============================================================================
# Dynamically finds and downloads the latest RxNorm Full Prescribe zip
# from the NLM RxNorm Files page if it doesn't already exist in out_folder.
#
# Extracts the NDC-to-RxCUI crosswalk from the RXNSAT.RRF file
# (ATN = 'NDC' and SAB = 'RXNORM').
#
# Writes a clean one-to-many mapping of NDC-11 → RxCUI to parquet
# so it can be queried later with DuckDB.
# =============================================================================

import pandas as pd
import zipfile
import requests
from pathlib import Path
from datetime import datetime
from bs4 import BeautifulSoup
from config import out_folder

out_path = Path(out_folder)
out_path.mkdir(parents=True, exist_ok=True)

zip_candidates = list(out_path.glob("RxNorm_full_prescribe_*.zip"))

if zip_candidates:
    zip_path = zip_candidates[0]
    print(f"Using existing: {zip_path.name}")
else:
    page_url = "https://www.nlm.nih.gov/research/umls/rxnorm/docs/rxnormfiles.html"
    resp = requests.get(page_url)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    link = soup.find("a", href=lambda href: href and "RxNorm_full_prescribe_" in href and href.endswith(".zip"))
    if not link:
        raise ValueError("Could not find latest RxNorm prescribe zip link.")

    url = link["href"]
    zip_name = url.split("/")[-1]
    zip_path = out_path / zip_name

    print(f"Downloading: {url}")
    r = requests.get(url, stream=True)
    r.raise_for_status()
    with open(zip_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024*1024):
            f.write(chunk)
    print("Download complete.")

load_date = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

with zipfile.ZipFile(zip_path) as z:
    sat = pd.read_csv(
        z.open("rrf/RXNSAT.RRF"),
        sep="|",
        header=None,
        dtype=str,
        usecols=[0, 8, 9, 10],
        names=["rxcui", "atn", "sab", "ndc11"]
    )

ndc_xwalk = (
    sat[(sat["atn"] == "NDC") & (sat["sab"] == "RXNORM")]
    [["ndc11", "rxcui"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

ndc_xwalk["load_date"] = load_date
ndc_xwalk.to_parquet(out_path / "ndc_xwalk.parquet", index=False)

print(ndc_xwalk.shape)
print(ndc_xwalk.head())
print("Wrote ndc_xwalk.parquet")

In [ ]:
# =============================================================================
# RxNorm Concepts
# =============================================================================
# Reads the previously downloaded RxNorm Full Prescribe zip
# (RxNorm_full_prescribe_*.zip) from the configured out_folder.
#
# Extracts preferred concept names from the RXNCONSO.RRF file
# for the most useful term types (SCD, SBD, GPCK, BPCK, IN, PIN, MIN, BN).
#
# Adds a human-readable term_type lookup and writes the result to parquet
# so it can be queried later with DuckDB.
# =============================================================================

import pandas as pd
import zipfile
from pathlib import Path
from datetime import datetime
from config import out_folder

out_path = Path(out_folder)

zip_candidates = list(out_path.glob("RxNorm_full_prescribe_*.zip"))
if not zip_candidates:
    raise FileNotFoundError(f"No RxNorm_full_prescribe_*.zip found in {out_folder}")

zip_path = zip_candidates[0]
print(f"Using: {zip_path.name}")

load_date = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

with zipfile.ZipFile(zip_path) as z:
    conso = pd.read_csv(
        z.open("rrf/RXNCONSO.RRF"),
        sep="|",
        header=None,
        dtype=str,
        usecols=[0, 11, 12, 13, 14],
        names=["rxcui", "suppress", "tty", "code", "str"]
    )

concepts = (
    conso[conso["tty"].isin(["SCD", "SBD", "GPCK", "BPCK", "IN", "PIN", "MIN", "BN"])]
    .drop_duplicates(subset="rxcui")
    [["rxcui", "str", "tty", "suppress"]]
    .rename(columns={"str": "name"})
    .reset_index(drop=True)
)

concepts["load_date"] = load_date

# TTY lookup
tty_lookup = pd.DataFrame({
    "tty": [
        "SCD", "SBD", "GPCK", "BPCK",
        "IN", "PIN", "BN", "DF",
        "SCDF", "SBDF", "SCDG", "SBDG",
        "MIN", "TMSY", "SY", "PSN"
    ],
    "term_type": [
        "Semantic Clinical Drug",
        "Semantic Branded Drug",
        "Generic Pack",
        "Branded Pack",
        "Ingredient",
        "Precise Ingredient",
        "Brand Name",
        "Dose Form",
        "Semantic Clinical Drug Form",
        "Semantic Branded Drug Form",
        "Semantic Clinical Drug Group",
        "Semantic Branded Drug Group",
        "Multiple Ingredients",
        "Tall Man Synonym",
        "Synonym",
        "Prescribable Name"
    ]
})

concepts = concepts.merge(tty_lookup, on="tty", how="left")

concepts.to_parquet(out_path / "concepts.parquet", index=False)

print(concepts.shape)
print(concepts.head())
print("Wrote concepts.parquet")

In [ ]:
# =============================================================================
# Load parquet tables and run the NDC + RxNorm join
# =============================================================================
# Expects the following parquet files to already exist in out_folder:
#   - ndc_package.parquet
#   - ndc_product.parquet
#   - ndc_xwalk.parquet
#   - concepts.parquet
# =============================================================================

import pandas as pd
import duckdb
from pathlib import Path
from config import out_folder

out_path = Path(out_folder)
con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE TABLE package AS 
SELECT * FROM read_parquet('{out_path / "ndc_package.parquet"}');

CREATE OR REPLACE TABLE product AS 
SELECT * FROM read_parquet('{out_path / "ndc_product.parquet"}');

CREATE OR REPLACE TABLE ndc_xwalk AS 
SELECT * FROM read_parquet('{out_path / "ndc_xwalk.parquet"}');

CREATE OR REPLACE TABLE concepts AS 
SELECT * FROM read_parquet('{out_path / "concepts.parquet"}');
""")

query = """
SELECT 
    p.ndc11,
    p.NDCPACKAGECODE,
    COALESCE(p.PACKAGEDESCRIPTION, '')          AS package_description,
    p.STARTMARKETINGDATE                        AS package_start_date,
    COALESCE(p.ENDMARKETINGDATE, '')            AS package_end_date,
    p.labeler_id,
    p.load_date                                 AS package_load_date,

    pr.PRODUCTNDC,
    COALESCE(pr.PROPRIETARYNAME, '')            AS proprietary_name,
    COALESCE(pr.NONPROPRIETARYNAME, '')         AS nonproprietary_name,
    COALESCE(pr.DOSAGEFORMNAME, '')             AS dosage_form,
    COALESCE(pr.ROUTENAME, '')                  AS route,
    COALESCE(pr.SUBSTANCENAME, '')              AS substance,
    COALESCE(pr.ACTIVE_NUMERATOR_STRENGTH, '')  AS strength,
    COALESCE(pr.ACTIVE_INGRED_UNIT, '')         AS strength_unit,
    COALESCE(pr.PHARM_CLASSES, '')              AS pharm_classes,
    COALESCE(pr.DEASCHEDULE, '')                AS dea_schedule,
    COALESCE(pr.LABELERNAME, '')                AS labeler_name,
    COALESCE(pr.MARKETINGCATEGORYNAME, '')      AS marketing_category,
    pr.STARTMARKETINGDATE                       AS product_start_date,
    COALESCE(pr.ENDMARKETINGDATE, '')           AS product_end_date,

    x.rxcui,
    x.load_date                                 AS xref_load_date,

    c.name                                      AS rxnorm_name,
    c.tty,
    c.term_type,
    c.suppress,
    c.load_date                                 AS concept_load_date

FROM package p
INNER JOIN product pr 
    ON p.PRODUCTID = pr.PRODUCTID
INNER JOIN ndc_xwalk x
    ON p.ndc11 = x.ndc11
INNER JOIN concepts c 
    ON x.rxcui = c.rxcui
"""

df = con.execute(query).df()

def widen_display(max_colwidth=80, width=1200):
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', width)
    pd.set_option('display.max_colwidth', max_colwidth)

widen_display()
print(df.head(10).to_string())
print(f"\nTotal rows: {len(df):,}")

excel_path = out_path / "ndc_rxnorm_join.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    df.to_excel(writer, index=False, sheet_name="ndc_rxnorm")
    worksheet = writer.sheets["ndc_rxnorm"]
    for col in worksheet.columns:
        worksheet.column_dimensions[col[0].column_letter].width = 50

print(f"Wrote {len(df):,} rows to {excel_path}")

In [ ]:
# =============================================================================
# ATC Dimension Builder
# -----------------------------------------------------------------------------
# Pulls Anatomical Therapeutic Chemical (ATC) classifications from the
# RxNav RxClass API for every unique RxCUI present in the existing NDC
# crosswalk. Results are written to atc_dimension.parquet for later joins
# in DuckDB.
#
# Source: https://rxnav.nlm.nih.gov/REST/rxclass/class/byRxcui
# Grain:  one row per RxCUI
# Output: atc_code, atc_name, atc_type
#
# Note: This makes one API call per unique RxCUI and includes a short pause
# every 50 calls to stay within rate limits. Runtime depends on the number
# of unique RxCUIs (can take 1–several hours for large sets).
# =============================================================================

import requests
import pandas as pd
import time
import duckdb
from pathlib import Path
from config import out_folder

out_path = Path(out_folder)

def get_atc(rxcui: str) -> dict:
    url = f"https://rxnav.nlm.nih.gov/REST/rxclass/class/byRxcui.json?rxcui={rxcui}&relaSource=ATC"
    try:
        r = requests.get(url, timeout=10)
        data = r.json()
        items = data.get("rxclassDrugInfoList", {}).get("rxclassDrugInfo", [])
        
        if not items:
            return {"rxcui": rxcui, "atc_code": None, "atc_name": None, "atc_type": None}
        
        first = items[0]["rxclassMinConceptItem"]
        return {
            "rxcui": rxcui,
            "atc_code": first.get("classId"),
            "atc_name": first.get("className"),
            "atc_type": first.get("classType")
        }
    except Exception:
        return {"rxcui": rxcui, "atc_code": None, "atc_name": None, "atc_type": None}

# --- Only pull RxCUIs that already exist in your crosswalk ---
con = duckdb.connect()
unique_rxcuis = con.execute("""
    SELECT DISTINCT rxcui 
    FROM ndc_xwalk 
    WHERE rxcui IS NOT NULL
""").df()["rxcui"].astype(str).tolist()

print(f"Unique RxCUIs to process: {len(unique_rxcuis):,}")

# --- Hit the API ---
atc_rows = []
for i, rxcui in enumerate(unique_rxcuis):
    atc_rows.append(get_atc(rxcui))
    if (i + 1) % 50 == 0:
        time.sleep(0.5)
        print(f"  Processed {i + 1}/{len(unique_rxcuis)}")

df_atc = pd.DataFrame(atc_rows)

# --- Write to configured out_path ---
out_file = out_path / "atc_dimension.parquet"
df_atc.to_parquet(out_file, index=False)
print(f"Wrote {len(df_atc):,} rows to {out_file}")
print(df_atc.head())

In [ ]:
import pandas as pd
import duckdb
from pathlib import Path
from config import out_folder

out_path = Path(out_folder)
con = duckdb.connect()

# Load all four tables
con.execute(f"""
CREATE OR REPLACE TABLE product AS 
SELECT * FROM read_parquet('{out_path / "ndc_product.parquet"}');

CREATE OR REPLACE TABLE package AS 
SELECT * FROM read_parquet('{out_path / "ndc_package.parquet"}');

CREATE OR REPLACE TABLE ndc_xwalk AS 
SELECT * FROM read_parquet('{out_path / "ndc_xwalk.parquet"}');

CREATE OR REPLACE TABLE atc AS 
SELECT * FROM read_parquet('{out_path / "atc_dimension.parquet"}');
""")

# Row counts
print(con.execute("SELECT COUNT(*) FROM product").fetchone()[0], "products")
print(con.execute("SELECT COUNT(*) FROM package").fetchone()[0], "packages")
print(con.execute("SELECT COUNT(*) FROM ndc_xwalk").fetchone()[0], "xwalk rows")
print(con.execute("SELECT COUNT(*) FROM atc").fetchone()[0], "ATC rows")

# Query
display(con.execute("""
SELECT
    p.PRODUCTNDC,
    p.PROPRIETARYNAME,
    p.NONPROPRIETARYNAME,
    pkg.NDCPACKAGECODE,
    pkg.ndc11,
    x.rxcui,
    a.atc_code,
    a.atc_name,
    a.atc_type
FROM product p
LEFT JOIN package pkg
    ON p.PRODUCTID = pkg.PRODUCTID
LEFT JOIN ndc_xwalk x
    ON pkg.ndc11 = x.ndc11
LEFT JOIN atc a
    ON x.rxcui = a.rxcui
LIMIT 10
""").fetchdf())

In [ ]:
import pandas as pd
import duckdb
from pathlib import Path
from config import out_folder

out_path = Path(out_folder)
con = duckdb.connect()

# Load all four tables
con.execute(f"""
CREATE OR REPLACE TABLE product AS 
SELECT * FROM read_parquet('{out_path / "ndc_product.parquet"}');

CREATE OR REPLACE TABLE package AS 
SELECT * FROM read_parquet('{out_path / "ndc_package.parquet"}');

CREATE OR REPLACE TABLE ndc_xwalk AS 
SELECT * FROM read_parquet('{out_path / "ndc_xwalk.parquet"}');

CREATE OR REPLACE TABLE atc AS 
SELECT * FROM read_parquet('{out_path / "atc_dimension.parquet"}');
""")

# Row counts
print(con.execute("SELECT COUNT(*) FROM product").fetchone()[0], "products")
print(con.execute("SELECT COUNT(*) FROM package").fetchone()[0], "packages")
print(con.execute("SELECT COUNT(*) FROM ndc_xwalk").fetchone()[0], "xwalk rows")
print(con.execute("SELECT COUNT(*) FROM atc").fetchone()[0], "ATC rows")

# Query
display(con.execute("""
SELECT
    COUNT(p.PRODUCTNDC) AS TotalRecords
    , a.atc_name
   
FROM product p
LEFT JOIN package pkg
    ON p.PRODUCTID = pkg.PRODUCTID
LEFT JOIN ndc_xwalk x
    ON pkg.ndc11 = x.ndc11
LEFT JOIN atc a
    ON x.rxcui = a.rxcui

WHERE a.atc_name IS NOT NULL
  AND a.atc_name != 'NAN'
  AND a.atc_name != ''

GROUP BY a.atc_name
ORDER BY TotalRecords DESC
LIMIT 10
""").fetchdf())

In [ ]:
# =============================================================================
# Build NADAC pricing benchmark dimension and write to Parquet
# =============================================================================
# Uses the CMS metastore API to always get the current download URL,
# then builds the NDC11 dimension.
# =============================================================================

import duckdb
from pathlib import Path
import urllib.request
import json
from config import out_folder

# ── PATHS ───────────────────────────────────────────────────────────────
NADAC_CSV = Path(out_folder) / "nadac.csv"
OUTPUT_PARQUET = Path(out_folder) / "dim_nadac_benchmark.parquet"

# Stable metastore endpoint for the current NADAC dataset
METASTORE_URL = "https://data.medicaid.gov/api/1/metastore/schemas/dataset/items/fbb83258-11c7-47f5-8b18-5f8e79f7e704"

# ── GET CURRENT DOWNLOAD URL ────────────────────────────────────────────
print("Fetching current NADAC download URL from CMS metastore...")
with urllib.request.urlopen(METASTORE_URL) as response:
    metadata = json.load(response)

download_url = metadata["distribution"][0]["downloadURL"]
print(f"Current download URL: {download_url}")

# ── DOWNLOAD IF NEEDED ──────────────────────────────────────────────────
if not NADAC_CSV.exists():
    print("Downloading latest NADAC file...")
    urllib.request.urlretrieve(download_url, NADAC_CSV)
    print(f"Saved to: {NADAC_CSV}")
else:
    print(f"Using existing file: {NADAC_CSV}")

# ── SETUP ───────────────────────────────────────────────────────────────
con = duckdb.connect()

print(f"\nBase folder : {out_folder}")
print(f"Input CSV   : {NADAC_CSV}")
print(f"Output      : {OUTPUT_PARQUET}")

# ── LOAD + BUILD DIMENSION ──────────────────────────────────────────────
print("\nBuilding NDC11 benchmark dimension...")

sql = f"""
WITH raw AS (
    SELECT *
    FROM read_csv_auto('{NADAC_CSV.as_posix()}', header=true)
),

cleaned AS (
    SELECT
        -- Standardize to 11-digit NDC11 (zero-padded, no hyphens)
        LPAD(REPLACE(CAST(NDC AS VARCHAR), '-', ''), 11, '0') AS ndc11,

        CAST(NDC AS VARCHAR) AS ndc_raw,
        "NDC Description" AS ndc_description,
        "NADAC Per Unit" AS nadac_per_unit,
        "Effective Date" AS effective_date,
        "Pricing Unit" AS pricing_unit,
        "Pharmacy Type Indicator" AS pharmacy_type_indicator,
        OTC,
        "Explanation Code" AS explanation_code,
        "Classification for Rate Setting" AS classification_for_rate_setting,
        "Corresponding Generic Drug NADAC Per Unit" AS corresponding_generic_nadac,
        "Corresponding Generic Drug Effective Date" AS corresponding_generic_effective_date,
        "As of Date" AS as_of_date

    FROM raw
    WHERE NDC IS NOT NULL
),

-- Keep only the most recent effective price per NDC11
latest AS (
    SELECT *
    FROM cleaned
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY ndc11
        ORDER BY effective_date DESC, as_of_date DESC
    ) = 1
)

SELECT
    ndc11,
    ndc_raw,
    ndc_description,
    nadac_per_unit,
    effective_date,
    pricing_unit,
    pharmacy_type_indicator,
    OTC,
    explanation_code,
    classification_for_rate_setting,
    corresponding_generic_nadac,
    corresponding_generic_effective_date,
    as_of_date,
    CURRENT_TIMESTAMP AS loaded_at
FROM latest
ORDER BY ndc11
"""

# Write to Parquet
con.execute(f"COPY ({sql}) TO '{OUTPUT_PARQUET.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD)")

# ── SUMMARY ─────────────────────────────────────────────────────────────
summary = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT ndc11) AS unique_ndc11,
        MIN(effective_date) AS earliest_effective,
        MAX(effective_date) AS latest_effective,
        COUNT(*) FILTER (WHERE nadac_per_unit IS NULL) AS null_prices,
        COUNT(*) FILTER (WHERE LENGTH(ndc11) != 11) AS bad_ndc11_length
    FROM read_parquet('{OUTPUT_PARQUET.as_posix()}')
""").fetchdf()

print("\n✅ NADAC Benchmark Dimension written")
print(f"Path: {OUTPUT_PARQUET}")
print("\nSummary:")
print(summary.to_string(index=False))

print("\nSample rows:")
print(con.execute(f"SELECT * FROM read_parquet('{OUTPUT_PARQUET.as_posix()}') LIMIT 5").fetchdf().to_string(index=False))

con.close()
print("\nDone.")

In [ ]:
# Validation: All source NADAC CSV records were loaded. 
# Output Parquet row count matches expected after NDC11 cleaning and latest-price deduplication.
# 100% of retained records have valid 11-digit NDC11.

import pandas as pd
from pathlib import Path
from config import out_folder

NADAC_CSV = Path(out_folder) / "nadac.csv"

if not NADAC_CSV.exists():
    print(f"File not found: {NADAC_CSV}")
else:
    # Quick row count (excluding header)
    with open(NADAC_CSV, "r", encoding="utf-8") as f:
        row_count = sum(1 for _ in f) - 1
    
    print(f"File: {NADAC_CSV}")
    print(f"Rows (excluding header): {row_count:,}")
    
    # Optional: also show column count and a quick peek
    df = pd.read_csv(NADAC_CSV, nrows=5)
    print(f"Columns: {len(df.columns)}")
    print("\nColumns:")
    print(df.columns.tolist())

In [ ]:
# =============================================================================
# Validation: All source NADAC CSV records were loaded. 
# Output Parquet row count matches expected after NDC11 cleaning and latest-price deduplication.
# 100% of retained records have valid 11-digit NDC11.
# =============================================================================

import pandas as pd
from pathlib import Path
from config import out_folder

OUTPUT_PARQUET = Path(out_folder) / "dim_nadac_benchmark.parquet"

if not OUTPUT_PARQUET.exists():
    print(f"File not found: {OUTPUT_PARQUET}")
else:
    df = pd.read_parquet(OUTPUT_PARQUET)
    
    print(f"File: {OUTPUT_PARQUET}")
    print(f"Rows: {len(df):,}")
    print(f"Unique NDC11: {df['ndc11'].nunique():,}")
    print(f"Columns: {len(df.columns)}")
    
    print("\nColumns:")
    print(df.columns.tolist())
    
    print("\nNDC11 length check (should be all 11):")
    print(df['ndc11'].str.len().value_counts().sort_index())
    
    print("\nNull price check:")
    print(f"Null nadac_per_unit: {df['nadac_per_unit'].isna().sum():,}")

In [ ]:
# =============================================================================
# Package + Product + NADAC Join (with package-level cost estimate)
# =============================================================================
# Expects the following parquet files to already exist in out_folder:
#   - ndc_package.parquet
#   - ndc_product.parquet
#   - dim_nadac_benchmark.parquet
# =============================================================================

import pandas as pd
import duckdb
from pathlib import Path
from config import out_folder
import re

out_path = Path(out_folder)
con = duckdb.connect()

# ── Load tables ─────────────────────────────────────────────────────────────
con.execute(f"""
CREATE OR REPLACE TABLE package AS 
SELECT * FROM read_parquet('{out_path / "ndc_package.parquet"}');

CREATE OR REPLACE TABLE product AS 
SELECT * FROM read_parquet('{out_path / "ndc_product.parquet"}');

CREATE OR REPLACE TABLE nadac AS 
SELECT * FROM read_parquet('{out_path / "dim_nadac_benchmark.parquet"}');
""")

print("Tables loaded.")
print(f"Package rows:  {con.execute('SELECT COUNT(*) FROM package').fetchone()[0]:,}")
print(f"Product rows:  {con.execute('SELECT COUNT(*) FROM product').fetchone()[0]:,}")
print(f"NADAC rows:    {con.execute('SELECT COUNT(*) FROM nadac').fetchone()[0]:,}")

# ── Join ────────────────────────────────────────────────────────────────────
query = """
SELECT
    p.ndc11,
    p.NDCPACKAGECODE,
    COALESCE(p.PACKAGEDESCRIPTION, '')          AS package_description,
    p.STARTMARKETINGDATE                        AS package_start_date,
    COALESCE(p.ENDMARKETINGDATE, '')            AS package_end_date,
    p.labeler_id,
    p.load_date                                 AS package_load_date,

    pr.PRODUCTNDC,
    COALESCE(pr.PROPRIETARYNAME, '')            AS proprietary_name,
    COALESCE(pr.NONPROPRIETARYNAME, '')         AS nonproprietary_name,
    COALESCE(pr.DOSAGEFORMNAME, '')             AS dosage_form,
    COALESCE(pr.ROUTENAME, '')                  AS route,
    COALESCE(pr.SUBSTANCENAME, '')              AS substance,
    COALESCE(pr.ACTIVE_NUMERATOR_STRENGTH, '')  AS strength,
    COALESCE(pr.ACTIVE_INGRED_UNIT, '')         AS strength_unit,
    COALESCE(pr.PHARM_CLASSES, '')              AS pharm_classes,
    COALESCE(pr.DEASCHEDULE, '')                AS dea_schedule,
    COALESCE(pr.LABELERNAME, '')                AS labeler_name,
    COALESCE(pr.MARKETINGCATEGORYNAME, '')      AS marketing_category,
    pr.STARTMARKETINGDATE                       AS product_start_date,
    COALESCE(pr.ENDMARKETINGDATE, '')           AS product_end_date,

    n.nadac_per_unit,
    n.effective_date                            AS nadac_effective_date,
    n.pricing_unit                              AS nadac_pricing_unit,
    n.pharmacy_type_indicator,
    n.OTC,
    n.explanation_code,
    n.classification_for_rate_setting,
    n.corresponding_generic_nadac,
    n.as_of_date                                AS nadac_as_of_date

FROM package p
INNER JOIN product pr 
    ON p.PRODUCTID = pr.PRODUCTID
INNER JOIN nadac n
    ON p.ndc11 = n.ndc11
"""

df = con.execute(query).df()

# Round base NADAC per unit to 5 decimals (matches CMS website display)
df["nadac_per_unit"] = df["nadac_per_unit"].round(5)

# ── Improved quantity extraction ────────────────────────────────────────────
def extract_package_quantity(desc: str, pricing_unit: str) -> tuple[float | None, str]:
    if not isinstance(desc, str) or not desc.strip():
        return None, "no description"

    desc_upper = desc.upper()
    pricing_unit = (pricing_unit or "").strip().upper()

    pattern = r"(\d+(?:\.\d+)?)\s*(TABLET|CAPSULE|TAB|CAP|VIAL|SYRINGE|PEN|ML|GM|GRAM|G|EACH|EA|UNIT|PACK|BOTTLE|CARTRIDGE|PATCH|SUPPOSITORY)"
    matches = re.findall(pattern, desc_upper)

    if not matches:
        return None, "no quantity pattern found"

    unit_map = {
        "TABLET": "EA", "CAPSULE": "EA", "TAB": "EA", "CAP": "EA",
        "VIAL": "EA", "SYRINGE": "EA", "PEN": "EA", "EACH": "EA", "EA": "EA",
        "UNIT": "EA", "PACK": "EA", "BOTTLE": "EA", "CARTRIDGE": "EA",
        "PATCH": "EA", "SUPPOSITORY": "EA",
        "ML": "ML",
        "GM": "GM", "GRAM": "GM", "G": "GM"
    }

    candidates = []
    for qty_str, unit_word in matches:
        mapped = unit_map.get(unit_word)
        if mapped:
            candidates.append((float(qty_str), unit_word, mapped))

    if not candidates:
        return None, "no mappable units"

    matching = [c for c in candidates if c[2] == pricing_unit]

    if matching:
        if pricing_unit in ("ML", "GM"):
            best = min(matching, key=lambda x: x[0])
        else:
            best = matching[0]
        return best[0], f"ok ({best[1]})"

    first = candidates[0]
    return None, f"unit mismatch (desc={first[1]}→{first[2]}, nadac={pricing_unit})"

# Apply
results = df.apply(
    lambda row: extract_package_quantity(row["package_description"], row["nadac_pricing_unit"]),
    axis=1,
    result_type="expand"
)
results.columns = ["package_qty", "reason"]

df = pd.concat([df, results], axis=1)

# Calculate estimate and round to 5 decimals
df["package_nadac_estimate"] = df.apply(
    lambda row: round(row["nadac_per_unit"] * row["package_qty"], 5)
    if pd.notna(row["package_qty"]) and pd.notna(row["nadac_per_unit"])
    else None,
    axis=1
)

# ── Reorder columns: calculated fields together, then description ───────────
calc_cols = [
    "ndc11",
    "nadac_pricing_unit",
    "package_qty",
    "nadac_per_unit",
    "package_nadac_estimate",
    "reason",
    "package_description"
]

# Keep all other columns after the calculated block
other_cols = [c for c in df.columns if c not in calc_cols]
df = df[calc_cols + other_cols]

# Summary
ok_count = df["reason"].str.startswith("ok").sum()
print(f"\nJoined rows: {len(df):,}")
print(f"Unique NDC11: {df['ndc11'].nunique():,}")
print(f"Package cost estimated: {ok_count:,} ({ok_count / len(df):.1%})")
print(f"Needs review: {len(df) - ok_count:,}")

def widen_display(max_colwidth=90, width=1800):
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", width)
    pd.set_option("display.max_colwidth", max_colwidth)

widen_display()
print(df[calc_cols].head(15).to_string())

# Write to parquet
output_file = out_path / "ndc_package_product_nadac.parquet"
df.to_parquet(output_file, index=False)
print(f"\nWrote {len(df):,} rows to {output_file}")

con.close()
print("Done.")

In [ ]:
# =============================================================================
# TEMP REVIEW EXPORT – Package + Product + NADAC to Excel
# =============================================================================
# Run this after the join cell. Delete this cell once review is done.
# =============================================================================

import pandas as pd
import duckdb
from pathlib import Path
from config import out_folder
from openpyxl import load_workbook

out_path = Path(out_folder)

# Prefer the already-written parquet if it exists; otherwise re-run the join
parquet_file = out_path / "ndc_package_product_nadac.parquet"

if parquet_file.exists():
    print(f"Reading existing parquet: {parquet_file}")
    df = pd.read_parquet(parquet_file)
else:
    print("Parquet not found – re-running join (this will take a moment)...")
    con = duckdb.connect()
    con.execute(f"""
    CREATE OR REPLACE TABLE package AS 
    SELECT * FROM read_parquet('{out_path / "ndc_package.parquet"}');

    CREATE OR REPLACE TABLE product AS 
    SELECT * FROM read_parquet('{out_path / "ndc_product.parquet"}');

    CREATE OR REPLACE TABLE nadac AS 
    SELECT * FROM read_parquet('{out_path / "dim_nadac_benchmark.parquet"}');
    """)

    query = """
    SELECT
        p.ndc11,
        p.NDCPACKAGECODE,
        COALESCE(p.PACKAGEDESCRIPTION, '')          AS package_description,
        p.STARTMARKETINGDATE                        AS package_start_date,
        COALESCE(p.ENDMARKETINGDATE, '')            AS package_end_date,
        p.labeler_id,
        p.load_date                                 AS package_load_date,

        pr.PRODUCTNDC,
        COALESCE(pr.PROPRIETARYNAME, '')            AS proprietary_name,
        COALESCE(pr.NONPROPRIETARYNAME, '')         AS nonproprietary_name,
        COALESCE(pr.DOSAGEFORMNAME, '')             AS dosage_form,
        COALESCE(pr.ROUTENAME, '')                  AS route,
        COALESCE(pr.SUBSTANCENAME, '')              AS substance,
        COALESCE(pr.ACTIVE_NUMERATOR_STRENGTH, '')  AS strength,
        COALESCE(pr.ACTIVE_INGRED_UNIT, '')         AS strength_unit,
        COALESCE(pr.PHARM_CLASSES, '')              AS pharm_classes,
        COALESCE(pr.DEASCHEDULE, '')                AS dea_schedule,
        COALESCE(pr.LABELERNAME, '')                AS labeler_name,
        COALESCE(pr.MARKETINGCATEGORYNAME, '')      AS marketing_category,
        pr.STARTMARKETINGDATE                       AS product_start_date,
        COALESCE(pr.ENDMARKETINGDATE, '')           AS product_end_date,

        n.nadac_per_unit,
        n.effective_date                            AS nadac_effective_date,
        n.pricing_unit                              AS nadac_pricing_unit,
        n.pharmacy_type_indicator,
        n.OTC,
        n.explanation_code,
        n.classification_for_rate_setting,
        n.corresponding_generic_nadac,
        n.as_of_date                                AS nadac_as_of_date

    FROM package p
    INNER JOIN product pr ON p.PRODUCTID = pr.PRODUCTID
    INNER JOIN nadac n ON p.ndc11 = n.ndc11
    """
    df = con.execute(query).df()
    con.close()

print(f"Rows to export: {len(df):,}")

# ── Write Excel ─────────────────────────────────────────────────────────────
excel_path = out_path / "ndc_package_product_nadac_REVIEW.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    df.to_excel(writer, index=False, sheet_name="nadac_review")
    worksheet = writer.sheets["nadac_review"]
    for col in worksheet.columns:
        worksheet.column_dimensions[col[0].column_letter].width = 20

# Force NDC-related columns to text
def force_text_columns(filepath, columns):
    wb = load_workbook(filepath)
    ws = wb.active
    headers = [cell.value for cell in ws[1]]
    for col_name in columns:
        if col_name in headers:
            col_idx = headers.index(col_name) + 1
            for cell in ws.iter_cols(min_col=col_idx, max_col=col_idx, min_row=2):
                for c in cell:
                    c.number_format = "@"
                    if c.value is not None:
                        c.value = str(c.value)
    wb.save(filepath)

force_text_columns(
    excel_path,
    ["ndc11", "NDCPACKAGECODE", "PRODUCTNDC", "labeler_id"]
)

print(f"Wrote review file: {excel_path}")
print("Done. You can delete this cell after reviewing.")

In [ ]:
# =============================================================================
# NADAC Historical Load (One-Time Combined Parquet)
# =============================================================================
# Objective
# ---------
# Download selected yearly CMS NADAC datasets and combine them into a single
# historical parquet for review and analysis.
#
# Source
# ------
# CMS data.medicaid.gov yearly NADAC datasets
# Metastore endpoint supplies the current download URL for each dataset.
#
# Output
# ------
# dim_nadac_historical.parquet  (in out_folder from config.py)
#
# Notes
# -----
# - Retains all weekly rates (no deduplication).
# - Adds a year column.
# - Reports source row counts and final parquet row count.
# - Update the YEARS dictionary with the dataset IDs you want included.
# =============================================================================

import duckdb
from pathlib import Path
import urllib.request
import json
from config import out_folder

# ── YEARS TO INCLUDE ────────────────────────────────────────────────────────
# To get a dataset ID for any year:
# 1. Go to https://data.medicaid.gov
# 2. Search for "NADAC YYYY" (e.g. NADAC 2024)
# 3. Open the dataset page
# 4. The Identifier (UUID) is shown under Additional Information
#    Use that UUID as the value in the key-value pair below.
#
# YEARS = {
#     "2013": "PLACEHOLDER_2013",
#     "2014": "PLACEHOLDER_2014",
#     "2015": "PLACEHOLDER_2015",
#     "2016": "PLACEHOLDER_2016",
#     "2017": "PLACEHOLDER_2017",
#     "2018": "PLACEHOLDER_2018",
#     "2019": "PLACEHOLDER_2019",
#     "2020": "PLACEHOLDER_2020",
#     "2021": "PLACEHOLDER_2021",
#     "2022": "PLACEHOLDER_2022",
#     "2023": "PLACEHOLDER_2023",
#     "2024": "PLACEHOLDER_2024",
#     "2025": "f38d0706-1239-442c-a3cc-40ef1b686ac0",
#     "2026": "fbb83258-11c7-47f5-8b18-5f8e79f7e704",
# }

YEARS = {
    "2025": "f38d0706-1239-442c-a3cc-40ef1b686ac0",
    "2026": "fbb83258-11c7-47f5-8b18-5f8e79f7e704",
}

out_path = Path(out_folder)
out_path.mkdir(parents=True, exist_ok=True)
OUTPUT_PARQUET = out_path / "dim_nadac_historical.parquet"

con = duckdb.connect()
total_source_rows = 0
union_parts = []

print(f"Output: {OUTPUT_PARQUET}")
print(f"Years: {list(YEARS.keys())}\n")

for year, dataset_id in YEARS.items():
    print(f"── {year} ──")
    metastore_url = f"https://data.medicaid.gov/api/1/metastore/schemas/dataset/items/{dataset_id}"

    with urllib.request.urlopen(metastore_url) as resp:
        metadata = json.load(resp)
    download_url = metadata["distribution"][0]["downloadURL"]

    csv_path = out_path / f"nadac_{year}.csv"

    if not csv_path.exists():
        print("Downloading...")
        urllib.request.urlretrieve(download_url, csv_path)
    else:
        print("Using existing CSV.")

    with open(csv_path, "r", encoding="utf-8") as f:
        rows = sum(1 for _ in f) - 1
    total_source_rows += rows
    print(f"Source rows: {rows:,}")

    con.execute(f"""
        CREATE OR REPLACE TEMP TABLE y{year} AS
        SELECT
            '{year}' AS year,
            LPAD(REPLACE(CAST(NDC AS VARCHAR), '-', ''), 11, '0') AS ndc11,
            CAST(NDC AS VARCHAR) AS ndc_raw,
            "NDC Description" AS ndc_description,
            "NADAC Per Unit" AS nadac_per_unit,
            "Effective Date" AS effective_date,
            "Pricing Unit" AS pricing_unit,
            "Pharmacy Type Indicator" AS pharmacy_type_indicator,
            OTC,
            "Explanation Code" AS explanation_code,
            "Classification for Rate Setting" AS classification_for_rate_setting,
            "Corresponding Generic Drug NADAC Per Unit" AS corresponding_generic_nadac,
            "Corresponding Generic Drug Effective Date" AS corresponding_generic_effective_date,
            "As of Date" AS as_of_date,
            CURRENT_TIMESTAMP AS loaded_at
        FROM read_csv_auto('{csv_path.as_posix()}', header=true)
        WHERE NDC IS NOT NULL
    """)
    union_parts.append(f"SELECT * FROM y{year}")

if union_parts:
    print("\nWriting combined parquet...")
    con.execute(f"COPY ({' UNION ALL '.join(union_parts)}) TO '{OUTPUT_PARQUET.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD)")

    summary = con.execute(f"""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT ndc11) AS unique_ndc11,
            MIN(effective_date) AS earliest,
            MAX(effective_date) AS latest,
            COUNT(DISTINCT year) AS years
        FROM read_parquet('{OUTPUT_PARQUET.as_posix()}')
    """).fetchdf()

    print(f"\n✅ Done")
    print(f"Source CSV rows total : {total_source_rows:,}")
    print(f"Parquet rows          : {summary['total_rows'].iloc[0]:,}")
    print(summary.to_string(index=False))

con.close()
print("\nHistorical load complete.")

In [ ]:
# =============================================================================
# Package + Product + Historical NADAC Join (Single NDC)
# =============================================================================
# Expects the following parquet files to already exist in out_folder:
#   - ndc_package.parquet
#   - ndc_product.parquet
#   - dim_nadac_historical.parquet
# =============================================================================

import duckdb
from pathlib import Path
from config import out_folder

out_path = Path(out_folder)
con = duckdb.connect()

# ── Load tables ─────────────────────────────────────────────────────────────
con.execute(f"""
CREATE OR REPLACE TABLE package AS 
SELECT * FROM read_parquet('{out_path / "ndc_package.parquet"}');

CREATE OR REPLACE TABLE product AS 
SELECT * FROM read_parquet('{out_path / "ndc_product.parquet"}');

CREATE OR REPLACE TABLE nadac_hist AS 
SELECT * FROM read_parquet('{out_path / "dim_nadac_historical.parquet"}');
""")

print("Tables loaded.")
print(f"Package rows:     {con.execute('SELECT COUNT(*) FROM package').fetchone()[0]:,}")
print(f"Product rows:     {con.execute('SELECT COUNT(*) FROM product').fetchone()[0]:,}")
print(f"NADAC hist rows:  {con.execute('SELECT COUNT(*) FROM nadac_hist').fetchone()[0]:,}")

# ── Join (change the NDC11 in the WHERE clause as needed) ──────────────────
query = """
SELECT
    p.ndc11,
    COALESCE(p.PACKAGEDESCRIPTION, '')          AS package_description,
    COALESCE(pr.PROPRIETARYNAME, '')            AS proprietary_name,
    COALESCE(pr.NONPROPRIETARYNAME, '')         AS nonproprietary_name,
    COALESCE(pr.DOSAGEFORMNAME, '')             AS dosage_form,
    COALESCE(pr.ROUTENAME, '')                  AS route,
    COALESCE(pr.ACTIVE_NUMERATOR_STRENGTH, '')  AS strength,
    COALESCE(pr.ACTIVE_INGRED_UNIT, '')         AS strength_unit,
    COALESCE(pr.LABELERNAME, '')                AS labeler_name,

    n.year,
    n.nadac_per_unit,
    n.effective_date,
    n.pricing_unit                              AS nadac_pricing_unit,
    n.pharmacy_type_indicator,
    n.OTC,
    n.explanation_code,
    n.classification_for_rate_setting,
    n.as_of_date

FROM package p
INNER JOIN product pr 
    ON p.PRODUCTID = pr.PRODUCTID
INNER JOIN nadac_hist n
    ON p.ndc11 = n.ndc11
WHERE p.ndc11 = '00002143380'          -- << change this NDC11
ORDER BY n.effective_date DESC, n.as_of_date DESC
"""

df = con.execute(query).df()

print(f"\nRows returned: {len(df):,}")
print(df.to_string(index=False))

con.close()
print("\nDone.")

In [ ]:
# =============================================================================
# Quick review of dim_nadac_historical.parquet
# =============================================================================

import duckdb
from pathlib import Path
from config import out_folder

out_path = Path(out_folder)
parquet_file = out_path / "dim_nadac_historical.parquet"

con = duckdb.connect()

print(f"File: {parquet_file}\n")

# Summary
summary = con.execute(f"""
    SELECT
        COUNT(*)                          AS total_rows,
        COUNT(DISTINCT ndc11)             AS unique_ndc11,
        COUNT(DISTINCT year)              AS years,
        MIN(effective_date)               AS earliest_effective,
        MAX(effective_date)               AS latest_effective
    FROM read_parquet('{parquet_file.as_posix()}')
""").fetchdf()

print("Summary:")
print(summary.to_string(index=False))

# Sample rows
print("\nSample rows:")
sample = con.execute(f"""
    SELECT 
        year,
        ndc11,
        ndc_description,
        nadac_per_unit,
        effective_date,
        pricing_unit,
        as_of_date
    FROM read_parquet('{parquet_file.as_posix()}')
    ORDER BY effective_date DESC
    LIMIT 10
""").fetchdf()

print(sample.to_string(index=False))

con.close()

In [ ]:
# =============================================================================
# FDA Orange Book Load
# =============================================================================
# Objective
# ---------
# Download the current FDA Orange Book data files and convert the three
# tables (products, patents, exclusivity) into parquet.
#
# Source
# ------
# https://www.fda.gov/media/76860/download?attachment
# (Updated monthly. Unzips into products.txt, patent.txt, exclusivity.txt)
#
# Output
# ------
# orangebook_products.parquet
# orangebook_patents.parquet
# orangebook_exclusivity.parquet
# =============================================================================

import duckdb
import zipfile
import urllib.request
import ssl
from pathlib import Path
from config import out_folder

out_path = Path(out_folder)
out_path.mkdir(parents=True, exist_ok=True)

ZIP_URL = "https://www.fda.gov/media/76860/download?attachment"
ZIP_PATH = out_path / "orangebook.zip"
EXTRACT_DIR = out_path / "orangebook_raw"
EXTRACT_DIR.mkdir(exist_ok=True)

# ── Download (SSL verification bypassed) ────────────────────────────────────
if not ZIP_PATH.exists():
    print("Downloading Orange Book ZIP...")
    ctx = ssl._create_unverified_context()
    with urllib.request.urlopen(ZIP_URL, context=ctx) as response:
        with open(ZIP_PATH, "wb") as out_file:
            out_file.write(response.read())
    print(f"Saved to: {ZIP_PATH}")
else:
    print(f"Using existing ZIP: {ZIP_PATH}")

# ── Extract ─────────────────────────────────────────────────────────────────
print("Extracting...")
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_DIR)

# ── Load into DuckDB and write parquet ──────────────────────────────────────
con = duckdb.connect()

files = {
    "products": EXTRACT_DIR / "products.txt",
    "patents": EXTRACT_DIR / "patent.txt",
    "exclusivity": EXTRACT_DIR / "exclusivity.txt",
}

for name, file_path in files.items():
    if not file_path.exists():
        print(f"Warning: {file_path} not found, skipping.")
        continue

    print(f"\n── {name.upper()} ──")

    # Count source rows before load
    with open(file_path, "r", encoding="latin-1") as f:
        source_rows = sum(1 for _ in f) - 1  # subtract header
    print(f"Source rows     : {source_rows:,}")

    out_file = out_path / f"orangebook_{name}.parquet"

    con.execute(f"""
        COPY (
            SELECT *, CURRENT_TIMESTAMP AS loaded_at
            FROM read_csv_auto(
                '{file_path.as_posix()}',
                delim='~',
                header=true,
                encoding='latin-1'
            )
        ) TO '{out_file.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD)
    """)

    # Count after load
    parquet_rows = con.execute(
        f"SELECT COUNT(*) FROM read_parquet('{out_file.as_posix()}')"
    ).fetchone()[0]
    print(f"Parquet rows    : {parquet_rows:,}")
    print(f"Output          : {out_file.name}")

con.close()
print("\nOrange Book load complete.")

In [ ]:
# =============================================================================
# Orange Book — Therapeutic Equivalence View
# =============================================================================
# Pharmacist use case
# -------------------
# Pharmacists use the Orange Book primarily for generic substitution.
# The Therapeutic Equivalence (TE) code indicates whether a generic is
# considered interchangeable with the Reference Listed Drug (RLD).
#
#   AB (or AA, AN, etc.) = therapeutically equivalent → substitutable
#   BX                   = not therapeutically equivalent → not substitutable
#
# This query joins the three Orange Book tables and surfaces the fields
# a pharmacist (or pharmacy system) would typically review.
#
# Expects these parquet files in out_folder:
#   - orangebook_products.parquet
#   - orangebook_patents.parquet
#   - orangebook_exclusivity.parquet
# =============================================================================

import duckdb
from pathlib import Path
from config import out_folder

out_path = Path(out_folder)
con = duckdb.connect()

# ── Load tables ─────────────────────────────────────────────────────────────
con.execute(f"""
CREATE OR REPLACE TABLE ob_products AS 
SELECT * FROM read_parquet('{out_path / "orangebook_products.parquet"}');

CREATE OR REPLACE TABLE ob_patents AS 
SELECT * FROM read_parquet('{out_path / "orangebook_patents.parquet"}');

CREATE OR REPLACE TABLE ob_exclusivity AS 
SELECT * FROM read_parquet('{out_path / "orangebook_exclusivity.parquet"}');
""")

print("Tables loaded.")
print(f"OB Products:    {con.execute('SELECT COUNT(*) FROM ob_products').fetchone()[0]:,}")
print(f"OB Patents:     {con.execute('SELECT COUNT(*) FROM ob_patents').fetchone()[0]:,}")
print(f"OB Exclusivity: {con.execute('SELECT COUNT(*) FROM ob_exclusivity').fetchone()[0]:,}")

# ── Therapeutic Equivalence Query ───────────────────────────────────────────
# Change the WHERE clause to a different trade name or application number
query = """
SELECT
    p."Trade_Name"                          AS trade_name,
    p."Ingredient"                          AS ingredient,
    p."DF;Route"                            AS dosage_form_route,
    p."Strength"                            AS strength,
    p."Appl_Type"                           AS appl_type,
    p."Appl_No"                             AS appl_no,
    p."Product_No"                          AS product_no,
    p."TE_Code"                             AS te_code,
    CASE 
        WHEN p."TE_Code" LIKE 'A%' THEN 'Substitutable'
        WHEN p."TE_Code" LIKE 'B%' THEN 'Not substitutable'
        ELSE 'Review'
    END                                     AS substitution_status,
    p."RLD"                                 AS rld,
    p."RS"                                  AS rs,
    p."Type"                                AS type,
    p."Applicant_Full_Name"                 AS applicant,

    pat."Patent_No"                         AS patent_no,
    pat."Patent_Expire_Date_Text"           AS patent_expire_date,

    ex."Exclusivity_Code"                   AS exclusivity_code,
    ex."Exclusivity_Date"                   AS exclusivity_date

FROM ob_products p
LEFT JOIN ob_patents pat
    ON p."Appl_No" = pat."Appl_No"
   AND p."Product_No" = pat."Product_No"
LEFT JOIN ob_exclusivity ex
    ON p."Appl_No" = ex."Appl_No"
   AND p."Product_No" = ex."Product_No"

WHERE UPPER(p."Trade_Name") = 'ABILIFY'     -- << change trade name
ORDER BY p."Strength", p."TE_Code", pat."Patent_No"
LIMIT 100
"""

df = con.execute(query).df()

print(f"\nRows returned: {len(df):,}")
print(df.to_string(index=False))

con.close()
print("\nDone.")